In [2]:
!pip install -q \
transformers \
accelerate \
openpyxl \
scikit-learn
!pip install -q sentencepiece

In [3]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [4]:
from google.colab import files

uploaded = files.upload()

Saving sinhala_youtube_comments_for_annotation.xlsx to sinhala_youtube_comments_for_annotation.xlsx


In [5]:
df = pd.read_excel(
    "/content/sinhala_youtube_comments_for_annotation.xlsx"
)

In [6]:
print(df.columns)

Index(['id', 'comment', 'label'], dtype='object')


In [7]:
df = df[['comment', 'label']]

In [8]:
df.dropna(inplace=True)

In [9]:
df['label'] = df['label'].astype(int)

In [10]:
print(df.head())

print(df['label'].value_counts())

                                             comment  label
0                                           පියතුමනි      0
1                                       නියම යි❤❤❤❤❤      0
2                              හිච්චගේ මුණ තමා maru😂      0
3  සානක අයියගෙ වීඩියෝ බලනවා නම් හිනා වෙන්න පුලුවන...      1
4  අයියා වෙඩින් මුද්ද නම් දාගෙනම නේද ටික් ටොක් කර...      0
label
0    1577
1    1471
Name: count, dtype: int64


In [11]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['comment'].astype(str).tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [12]:
MODEL_NAME = "ai4bharat/IndicBERTv2-MLM-only"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those p

In [13]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=256
)

In [14]:
class SarcasmDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item['labels'] = torch.tensor(
            self.labels[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

In [15]:
train_dataset = SarcasmDataset(
    train_encodings,
    train_labels
)

test_dataset = SarcasmDataset(
    test_encodings,
    test_labels
)

In [16]:
def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='macro'
    )

    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [22]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=10,

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    warmup_ratio=0.1,

    weight_decay=0.01,

    logging_steps=10,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    greater_is_better=True,

    save_total_limit=1
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [23]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics
)

In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.731890,0.694043,0.518033,0.341253,0.259016,0.500000
2,0.676980,0.683000,0.516393,0.340541,0.258621,0.498418
3,0.685486,0.681727,0.629508,0.629409,0.629569,0.629736
4,0.603701,0.665315,0.608197,0.597598,0.631850,0.614850
5,0.704687,0.645080,0.639344,0.639309,0.640787,0.640532
6,0.560216,0.665307,0.655738,0.655367,0.655341,0.655408
7,0.541561,0.636622,0.668852,0.668050,0.668383,0.667948
8,0.555421,0.629137,0.672131,0.669913,0.672648,0.670046
9,0.565538,0.694924,0.668852,0.663348,0.673252,0.665106
10,0.451849,0.714897,0.655738,0.654251,0.655428,0.654224


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3050, training_loss=0.6046434603362787, metrics={'train_runtime': 2931.8351, 'train_samples_per_second': 8.316, 'train_steps_per_second': 1.04, 'total_flos': 3207323764838400.0, 'train_loss': 0.6046434603362787, 'epoch': 10.0})

In [25]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
0.451849,0.629137,10,0.672131,0.669913,0.672648,0.670046


{'eval_loss': 0.6291374564170837, 'eval_accuracy': 0.6721311475409836, 'eval_f1': 0.66991341991342, 'eval_precision': 0.6726481902430494, 'eval_recall': 0.6700464996125033}


In [26]:
predictions = trainer.predict(test_dataset)

preds = predictions.predictions.argmax(-1)

cm = confusion_matrix(test_labels, preds)

print(cm)

[[230  86]
 [114 180]]


In [27]:
print(
    classification_report(
        test_labels,
        preds
    )
)

              precision    recall  f1-score   support

           0       0.67      0.73      0.70       316
           1       0.68      0.61      0.64       294

    accuracy                           0.67       610
   macro avg       0.67      0.67      0.67       610
weighted avg       0.67      0.67      0.67       610



In [28]:
model.save_pretrained("./indicbert_model")
tokenizer.save_pretrained("./indicbert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./indicbert_model/tokenizer_config.json', './indicbert_model/tokenizer.json')

In [29]:
!zip -r sarcasm_model.zip sarcasm_model

	zip warning: name not matched: sarcasm_model

zip error: Nothing to do! (try: zip -r sarcasm_model.zip . -i sarcasm_model)


In [33]:
text = "හරිම ලස්සන වැඩක් කරලා 😂"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

inputs = {key: value.to(device) for key, value in inputs.items()}

outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1)

print("Prediction:", prediction.item())

if prediction.item() == 1:
    print("Sarcastic")
else:
    print("Non-Sarcastic")

Prediction: 1
Sarcastic
